In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg,col,first,when,udf,regexp_replace
from pyspark.sql.types import StringType
ss = SparkSession.builder.config("spark.jars", "/home/jrodarte/postgresql-42.7.3.jar").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/11 20:54:30 WARN Utils: Your hostname, jrodarte-ThinkPad-T480, resolves to a loopback address: 127.0.1.1; using 192.168.100.13 instead (on interface wlp3s0)
25/08/11 20:54:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/08/11 20:54:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
dfMovimientos = ss.read.format("csv").options(header='true', inferSchema='true', delimiter=',').load("movimientos.csv")

In [3]:
dfMovimientos.show()

+------------+-------------------+-----+----------+-------+--------+--------------------+--------------------+
|Autorización|    Fecha operación| Tipo|Movimiento|Importe| Estatus|              Ref. 1|             Detalle|
+------------+-------------------+-----+----------+-------+--------+--------------------+--------------------+
|   103124028|31/07/2025 17:06:19|PAGOS|     PAGOS|  $0.00|APLICADO|PAGO DE: HARVEY RUIZ|Principal: 0.0000...|
|   103105326|31/07/2025 16:58:03|PAGOS|     PAGOS|  $0.01|APLICADO|     PAGO DE: GMORIN|Principal: 0.0000...|
|   103077947|31/07/2025 10:46:32|PAGOS|     PAGOS| $11.17|APLICADO|     PAGO DE: PORUGA|Principal: 6.9495...|
|   103077946|31/07/2025 10:46:32|PAGOS| RETENCION|  $0.00|APLICADO|       RETENCION ISR|                NULL|
|   103077945|31/07/2025 10:46:32|PAGOS| RETENCION|  $0.00|APLICADO|       RETENCION IVA|                NULL|
|   102992053|30/07/2025 13:46:03|PAGOS| RETENCION|  $0.00|APLICADO|       RETENCION ISR|                NULL|
|

In [8]:
dfMovimientos.select("Movimiento").distinct().show()

+----------+
|Movimiento|
+----------+
|    FONDEO|
|     PAGOS|
|  COMISION|
| RETENCION|
|     ABONO|
+----------+



In [10]:
dfTipoMovimientos = dfMovimientos.select(col("Movimiento").alias("tipomovimiento")).distinct()

In [11]:
dfTipoMovimientos.show()

+--------------+
|tipomovimiento|
+--------------+
|        FONDEO|
|         PAGOS|
|      COMISION|
|     RETENCION|
|         ABONO|
+--------------+



In [12]:
dfTipoMovimientos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.tipomovimientos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()